# Mamba — TaskTracker-only

Pipeline dung 470 TaskTracker sessions va ba split CSV co dinh cua nhom. Test bat buoc 69 sessions (15 label 1, 54 label 0). Notebook khong tao split moi. PasteTrace khong duoc su dung.

In [ ]:
!pip install -q -r requirements-mamba.txt
import torch
from mamba_ssm import Mamba
print('CUDA:', torch.cuda.is_available(), '| torch:', torch.__version__)
print('Mamba output:', Mamba(d_model=64)(torch.randn(2, 32, 64, device='cuda')).shape)

In [ ]:
!python -m src.data.build_sequences --min-events 1
!python -m src.data.make_splits  # validate only; never creates a split

In [ ]:
import pandas as pd
for name in ['train', 'validation', 'test']:
    frame = pd.read_csv(f'data/splits/tasktracker/{name}.csv')
    print(name, len(frame))

## Train + validation
Scaler chi fit train. Checkpoint va threshold duoc chon bang validation. Test split khong duoc nap trong qua trinh nay.

In [ ]:
!python -m src.models.mamba_model train \
    --d-model 64 --n-layers 2 --dropout 0.2 \
    --epochs 80 --lr 3e-4 --patience 10 \
    --batch-size 8 --max-len 1000 --seed 42

## Held-out TaskTracker test
Lenh nay tao `results/mamba/predictions.csv` va `results/mamba/metrics.json`.

In [ ]:
!python -m src.models.mamba_model test
import json, pandas as pd
display(pd.read_csv('results/mamba/predictions.csv').head())
with open('results/mamba/metrics.json', encoding='utf8') as f:
    metrics = json.load(f)
metrics